In [2]:
import sqlite3
from bs4 import BeautifulSoup
import requests
from fake_useragent import UserAgent

## 1. Создаём базу данных:
**Внимание! есть добавление в самом низу (1)**

In [3]:
con = sqlite3.connect('datapoems.db')  # подключение
cur = con.cursor()  # курсор

In [3]:
cur.execute("""
CREATE TABLE themes(
id INTEGER PRIMARY KEY AUTOINCREMENT,
theme TEXT,
vector_theme TEXT
)
""")

In [4]:
cur.execute("""
CREATE TABLE authors(
id INTEGER PRIMARY KEY AUTOINCREMENT,
name TEXT,
born INTEGER,
died INTEGER
)
""")

In [5]:
cur.execute("""
CREATE TABLE poems(
id INTEGER PRIMARY KEY AUTOINCREMENT,
poem TEXT,
theme_id INTEGER,
author_id INTEGER,
vector TEXT,
popularity INTEGER,
FOREIGN KEY (theme_id) REFERENCES themes (id),
FOREIGN KEY (author_id) REFERENCES authors (id)
)
""")

In [6]:
cur.execute("""
CREATE TABLE user(
id INTEGER PRIMARY KEY AUTOINCREMENT,
ask TEXT,
vector TEXT,
theme_user INTEGER,
answer_id INTEGER,
FOREIGN KEY (theme_user) REFERENCES themes (id),
FOREIGN KEY (answer_id) REFERENCES poems (id)
)
""")

In [7]:
con.commit()

In [9]:
moods = [("любовь",), ("жизнь",), ("женщина",), ("дружба",), ("смерть",), ("люди",), ("предательство",), ("счастье",), ("вино",), ("Бог",), ("деньги",)]
cur.executemany("""INSERT INTO themes(theme) VALUES (?)""", (moods))
con.commit()

In [10]:
cur.execute(f"""INSERT INTO authors VALUES (1, "Ома́р Хайя́м (Гия́с-ад-Ди́н Абу-ль-Фатх Ома́р ибн Ибрахи́м Хайя́м Нишапури́)", 1048, 1131)""")
con.commit()

## 2. Собираем стихотворения краулером и вставляем их в базу. Я также отмечала, если на сайте некоторые стихотворения были помечены, как относящиеся к теме (темы собраны в таблицу themes): это шло в столбец таблицы poems в theme_id. 
Собиралась, но пока что не реализовала векторизацию самих названий тем, чтобы с ними тоже сраванивать запросы пользователя и таким образом сужать множество для сравнений: это задача на будущее.

In [11]:
dict_of_urls = {"все": "https://stihionline.ru/rubai-omara-hajyama/",
                "любовь": "https://stihionline.ru/stihi-o-lyubvi-zarubezhnyh-poetov/rubai-omara-hajyama-o-lyubvi/",
                "жизнь": "https://stihionline.ru/stihi-o-zhizni-zarubezhnyh-poetov/rubai-o-zhizni-omara-hajyama/",
                "женщина": "https://stihionline.ru/stihi-o-zhenshhine-zarubezhnyh-poetov/rubai-omara-hajyama-o-zhenshhine/",
                "дружба": "https://stihionline.ru/stihi-o-druzhbe-zarubezhnyh-poetov/rubai-omara-hajyama-o-druzhbe/",
                "смерть": "https://stihionline.ru/stihi-o-smerti-zarubezhnyh-poetov/rubai-omara-hajyama-o-smerti/",
                "люди": "https://stihionline.ru/rubai-omara-hajyama-o-lyudyah/",
                "предательство": "https://stihionline.ru/rubai-omara-hajyama-o-predatelstve-i-izmene/",
                "счастье": "https://stihionline.ru/stihi-o-schaste-zarubezhnyh-poetov/rubai-omara-hajyama-o-schaste/",
                "вино": "https://stihionline.ru/rubai-omara-hajyama-o-vine/",
                "Бог": "https://stihionline.ru/rubai-omara-hajyama-o-boge/",
                "деньги": "https://stihionline.ru/rubai-omara-hajyama-o-dengah/"
                }

In [12]:
def parse_page(theme, x):
    session = requests.session()
    ua = UserAgent()
    ssyl = dict_of_urls[f"{theme}"]
    url = f"{ssyl}/page/{x}"
    req = session.get(url, headers={'User-Agent': ua.random})
    page = req.text
    soup = BeautifulSoup(page, 'html.parser')
    poems = soup.find_all('div', {"class": "content-data"})
    texts = []
    for x in poems:
        t = (x.text,)
        texts.append(t)
    return texts

In [13]:
def parse_theme(theme, n):
    full_texts = []
    for x in range(1, n+1):
        texts = parse_page(theme, x)
        full_texts.extend(texts)
    return full_texts

In [14]:
def parse_all():
    textsall = parse_theme("все", 42)
    dict_final = {}
    dict_final[1] = parse_theme("любовь", 9)
    dict_final[2] = parse_theme("жизнь", 5)
    dict_final[3] = parse_theme("женщина", 2)
    dict_final[4] = parse_theme("дружба", 2)
    dict_final[5] = parse_theme("смерть", 5)
    dict_final[6] = parse_theme("люди", 2)
    dict_final[7] = parse_theme("предательство", 2)
    dict_final[8] = parse_theme("счастье", 2)
    dict_final[9] = parse_theme("вино", 10)
    dict_final[10] = parse_theme("Бог", 5)
    dict_final[11] = parse_theme("деньги", 2)
    cur.executemany("""INSERT INTO poems (poem, theme_id, author_id) VALUES (?, NULL, 1)""", textsall)
    for key in dict_final:
        for i in dict_final[key]:
            cur.execute("UPDATE poems SET theme_id = ? WHERE poem = ?", (key, i[0]))
    con.commit()

In [15]:
parse_all()

## 3. Загружаем модель для векторизации. Я использовала модель, обученную на НКРЯ 2018-го года.

In [24]:
import gensim
import logging
import urllib.request

In [25]:
urllib.request.urlretrieve("https://rusvectores.org/static/models/rusvectores4/RNC/ruscorpora_upos_skipgram_300_5_2018.vec.gz", "ruscorpora_upos_skipgram_300_5_2018.vec.gz")

('ruscorpora_upos_skipgram_300_5_2018.vec.gz',
 <http.client.HTTPMessage at 0x20b6131d810>)

In [26]:
m = "ruscorpora_upos_skipgram_300_5_2018.vec.gz"
modelmine = gensim.models.KeyedVectors.load_word2vec_format(m, binary=False)

### 3.5. Далее следует функция очистки и лемматизации текста + приведения его к виду, который возможно загрузить в модель. Функция была взята [отсюда](https://github.com/akutuzov/webvectors/blob/master/preprocessing/rus_preprocessing_udpipe.py) (точнее, код был переведён в функцию, как это сделано [здесь](https://notebook.community/akutuzov/webvectors/preprocessing/rusvectores_tutorial)) и модифицирована мной под то, чтобы получать на вход не список, а строку: мне так было удобнее далее писать функции.

In [29]:
#!/usr/bin/env python3
# coding: utf-8

import sys
import os
import wget
import re
from ufal.udpipe import Model, Pipeline

"""
Этот скрипт принимает на вход необработанный русский текст 
(одно предложение на строку или один абзац на строку).
Он токенизируется, лемматизируется и размечается по частям речи с использованием UDPipe.
На выход подаётся последовательность разделенных пробелами лемм с частями речи 
("зеленый_ADJ трамвай_NOUN").
Их можно непосредственно использовать в моделях с RusVectōrēs (https://rusvectores.org).

Примеры запуска:
echo 'Мама мыла раму.' | python3 rus_preprocessing_udpipe.py
zcat large_corpus.txt.gz | python3 rus_preprocessing_udpipe.py | gzip > processed_corpus.txt.gz
"""


def num_replace(word):
    newtoken = "x" * len(word)
    return newtoken


def clean_token(token, misc):
    """
    :param token:  токен (строка)
    :param misc:  содержимое поля "MISC" в CONLLU (строка)
    :return: очищенный токен (строка)
    """
    out_token = token.strip().replace(" ", "")
    if token == "Файл" and "SpaceAfter=No" in misc:
        return None
    return out_token


def clean_lemma(lemma, pos):
    """
    :param lemma: лемма (строка)
    :param pos: часть речи (строка)
    :return: очищенная лемма (строка)
    """
    out_lemma = lemma.strip().replace(" ", "").replace("_", "").lower()
    if "|" in out_lemma or out_lemma.endswith(".jpg") or out_lemma.endswith(".png"):
        return None
    if pos != "PUNCT":
        if out_lemma.startswith("«") or out_lemma.startswith("»"):
            out_lemma = "".join(out_lemma[1:])
        if out_lemma.endswith("«") or out_lemma.endswith("»"):
            out_lemma = "".join(out_lemma[:-1])
        if (
            out_lemma.endswith("!")
            or out_lemma.endswith("?")
            or out_lemma.endswith(",")
            or out_lemma.endswith(".")
        ):
            out_lemma = "".join(out_lemma[:-1])
    return out_lemma


def list_replace(search, replacement, text):
    search = [el for el in search if el in text]
    for c in search:
        text = text.replace(c, replacement)
    return text


def unify_sym(text):  # принимает строку в юникоде
    text = list_replace(
        "\u00AB\u00BB\u2039\u203A\u201E\u201A\u201C\u201F\u2018\u201B\u201D\u2019",
        "\u0022",
        text,
    )

    text = list_replace(
        "\u2012\u2013\u2014\u2015\u203E\u0305\u00AF", "\u2003\u002D\u002D\u2003", text
    )

    text = list_replace("\u2010\u2011", "\u002D", text)

    text = list_replace(
        "\u2000\u2001\u2002\u2004\u2005\u2006\u2007\u2008\u2009\u200A\u200B\u202F\u205F\u2060\u3000",
        "\u2002",
        text,
    )

    text = re.sub("\u2003\u2003", "\u2003", text)
    text = re.sub("\t\t", "\t", text)

    text = list_replace(
        "\u02CC\u0307\u0323\u2022\u2023\u2043\u204C\u204D\u2219\u25E6\u00B7\u00D7\u22C5\u2219\u2062",
        ".",
        text,
    )

    text = list_replace("\u2217", "\u002A", text)

    text = list_replace("…", "...", text)

    text = list_replace("\u2241\u224B\u2E2F\u0483", "\u223D", text)

    text = list_replace("\u00C4", "A", text)  # латинская
    text = list_replace("\u00E4", "a", text)
    text = list_replace("\u00CB", "E", text)
    text = list_replace("\u00EB", "e", text)
    text = list_replace("\u1E26", "H", text)
    text = list_replace("\u1E27", "h", text)
    text = list_replace("\u00CF", "I", text)
    text = list_replace("\u00EF", "i", text)
    text = list_replace("\u00D6", "O", text)
    text = list_replace("\u00F6", "o", text)
    text = list_replace("\u00DC", "U", text)
    text = list_replace("\u00FC", "u", text)
    text = list_replace("\u0178", "Y", text)
    text = list_replace("\u00FF", "y", text)
    text = list_replace("\u00DF", "s", text)
    text = list_replace("\u1E9E", "S", text)

    currencies = list(
        "\u20BD\u0024\u00A3\u20A4\u20AC\u20AA\u2133\u20BE\u00A2\u058F\u0BF9\u20BC\u20A1\u20A0\u20B4\u20A7\u20B0\u20BF\u20A3\u060B\u0E3F\u20A9\u20B4\u20B2\u0192\u20AB\u00A5\u20AD\u20A1\u20BA\u20A6\u20B1\uFDFC\u17DB\u20B9\u20A8\u20B5\u09F3\u20B8\u20AE\u0192"
    )

    alphabet = list(
        '\t\n\r абвгдеёзжийклмнопрстуфхцчшщьыъэюяАБВГДЕЁЗЖИЙКЛМНОПРСТУФХЦЧШЩЬЫЪЭЮЯ,.[]{}()=+-−*&^%$#@!?~;:0123456789§/\|"abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ '
    )

    alphabet.append("'")

    allowed = set(currencies + alphabet)

    cleaned_text = [sym for sym in text if sym in allowed]
    cleaned_text = "".join(cleaned_text)

    return cleaned_text


def process(pipeline, text="Строка", keep_pos=True, keep_punct=False):
    # Если частеречные тэги не нужны (например, их нет в модели), выставьте pos=False
    # в этом случае на выход будут поданы только леммы
    # По умолчанию знаки пунктуации вырезаются. Чтобы сохранить их, выставьте punct=True

    entities = {"PROPN"}
    named = False
    memory = []
    mem_case = None
    mem_number = None
    tagged_propn = []

    # обрабатываем текст, получаем результат в формате conllu:
    processed = pipeline.process(text)

    # пропускаем строки со служебной информацией:
    content = [line for line in processed.split("\n") if not line.startswith("#")]

    # извлекаем из обработанного текста леммы, тэги и морфологические характеристики
    tagged = [w.split("\t") for w in content if w]

    for t in tagged:
        if len(t) != 10:
            continue
        (word_id, token, lemma, pos, xpos, feats, head, deprel, deps, misc) = t
        token = clean_token(token, misc)
        lemma = clean_lemma(lemma, pos)
        if not lemma or not token:
            continue
        if pos in entities:
            if "|" not in feats:
                tagged_propn.append("%s_%s" % (lemma, pos))
                continue
            morph = {el.split("=")[0]: el.split("=")[1] for el in feats.split("|")}
            if "Case" not in morph or "Number" not in morph:
                tagged_propn.append("%s_%s" % (lemma, pos))
                continue
            if not named:
                named = True
                mem_case = morph["Case"]
                mem_number = morph["Number"]
            if morph["Case"] == mem_case and morph["Number"] == mem_number:
                memory.append(lemma)
                if "SpacesAfter=\\n" in misc or "SpacesAfter=\s\\n" in misc:
                    named = False
                    past_lemma = "::".join(memory)
                    memory = []
                    tagged_propn.append(past_lemma + "_PROPN")
            else:
                named = False
                past_lemma = "::".join(memory)
                memory = []
                tagged_propn.append(past_lemma + "_PROPN")
                tagged_propn.append("%s_%s" % (lemma, pos))
        else:
            if not named:
                if (
                    pos == "NUM" and token.isdigit()
                ):  # Заменяем числа на xxxxx той же длины
                    lemma = num_replace(token)
                tagged_propn.append("%s_%s" % (lemma, pos))
            else:
                named = False
                past_lemma = "::".join(memory)
                memory = []
                tagged_propn.append(past_lemma + "_PROPN")
                tagged_propn.append("%s_%s" % (lemma, pos))

    if not keep_punct:
        tagged_propn = [word for word in tagged_propn if word.split("_")[1] != "PUNCT"]
    if not keep_pos:
        tagged_propn = [word.split("_")[0] for word in tagged_propn]
    return tagged_propn

def tag_ud(text='Текст нужно передать функции в виде строки!', modelfile='udpipe_syntagrus.model'):
    udpipe_model_url = 'https://rusvectores.org/static/models/udpipe_syntagrus.model'
    udpipe_filename = udpipe_model_url.split('/')[-1]

    if not os.path.isfile(modelfile):
        print('UDPipe model not found. Downloading...', file=sys.stderr)
        wget.download(udpipe_model_url)

    # print('\nLoading the model...', file=sys.stderr)
    model = Model.load(modelfile)
    process_pipeline = Pipeline(model, 'tokenize', Pipeline.DEFAULT, Pipeline.DEFAULT, 'conllu')

    # print('Processing input...', file=sys.stderr)
    # line = unify_sym(line.strip()) # здесь могла бы быть ваша функция очистки текста
    output = process(process_pipeline, text=text)
    return ' '.join(output)

## 4. Функция получения усреднённого вектора

In [32]:
y = get_comvect('На осле ехать —\nНогам покоя не знать;\nС двумя женами жить —\nУшам покоя не знать!\n')

In [30]:
import numpy as np
def get_comvect(text):
    textnew = [w for w in (tag_ud(text)).split()]
    vectorsfunc = []
    for w in textnew:
        try:
            vectorsfunc.append(modelmine.get_vector(w)[:10])
        except:
            continue       
    return np.mean(vectorsfunc, axis=0)

## 5. Проверка работы функции

In [34]:
from sklearn.metrics.pairwise import cosine_similarity

In [35]:
answer = get_comvect('Что такое человек?')

In [36]:
y

array([-0.0392559, -0.0212071,  0.0102202, -0.0613558,  0.0003234,
       -0.012833 ,  0.0149156, -0.0366338,  0.0048864,  0.0495674],
      dtype=float32)

In [37]:
answer

array([-0.061119,  0.0601  ,  0.079129, -0.034134,  0.113729,  0.055199,
        0.012795, -0.02478 , -0.079385, -0.028601], dtype=float32)

In [38]:
cosine_similarity(y.reshape(1, -1), answer.reshape(1, -1))

array([[0.13428126]], dtype=float32)

## 6. Сохраняем усреднённые вектора в таблицу заранее: чтобы потом не считать их каждый раз при запросе пользователя
Приходится делать это переводом в строки: база данных через sqlite3 не берёт массив нампай

In [40]:
# let's save mean vectors of the poems

import pandas as pd
query = "SELECT id, poem FROM poems"
df = pd.read_sql_query(query, con=con)
df

,id,poem
0,1,"Не делай зла — вернется бумерангом,\nНе плюй в..."
1,2,Дарить себя — не значит продавать.\nИ рядом сп...
2,3,"Не завидуй тому, кто силен и богат,\nза рассве..."
3,4,"Кто понял жизнь тот больше не спешит,\nСмакует..."
4,5,"Чтоб мудро жизнь прожить, знать надобно немало..."
...,...,...
486,487,"Все, что будет: и зло, и добро — пополам —\nПр..."
487,488,"Все — и зло и добро, что людская скрывает прир..."
488,489,В пути запуталась душа в добре и зле…\nЧист от...
489,490,"Вместо солнца весь мир озарить — не могу,\nВ т..."


#### Ячейка ниже -- самая долго работающая во всем prework, будьте осторожны. Работает до 45 минут: это векторизация 491 стиха, так что логично.

In [41]:
# creating vectors to all 

vectors = []
for x in df['poem']:
    addition = get_comvect(x)
    vectors.append(addition)

In [43]:
len(vectors)

491

In [71]:
a = " ".join(list(map(str, list(vectors[0]))))
b = " ".join(list(map(str, list(vectors[1]))))
a

'-0.0092478795 -0.035574116 0.020726638 -0.015780361 0.03166056 0.0032186015 -0.016294679 0.023374436 -0.026318641 0.0015474398'

In [80]:
# now we make vectors into format that our database is able to take and getting them to the correspondent table
vectorsforsave = {}
for x in range(len(vectors)):
    save = " ".join(list(map(str, list(vectors[x]))))
    vectorsforsave[x + 1] = save
for key in vectorsforsave:
    cur.execute("""
    UPDATE poems SET vector = ? WHERE id = ?
    """, (vectorsforsave[key], key))

In [81]:
con.commit()

## 7. Тестим всё вместе

In [121]:
# testing the process of getting the vector from the table and comparing it to the user's input
answer = "маска"
test = get_comvect(answer)

In [122]:
query = "SELECT id, vector FROM poems"
df = pd.read_sql_query(query, con=con)
df

,id,vector
0,1,-0.0092478795 -0.035574116 0.020726638 -0.0157...
1,2,-0.024054924 0.016276155 -0.0038047691 -0.0324...
2,3,-0.008320249 0.0025840828 -0.002242667 -0.0587...
3,4,-0.023275562 -0.010551225 0.0101969065 -0.0147...
4,5,-0.030326068 -0.014095 0.013723868 -0.05095206...
...,...,...
486,487,-0.014819933 -0.0265392 0.037992336 -0.0305137...
487,488,-0.027691638 -0.026864683 0.02613421 -0.004013...
488,489,-0.016210532 -0.028556734 0.013557867 -0.02558...
489,490,0.0017195886 -0.019688532 0.0027031763 -0.0269...


In [123]:
c = 0.0
for x in df["vector"]:
    compare = np.asarray(list((map(float, x.split(" ")))))
    if cosine_similarity(test.reshape(1, -1), compare.reshape(1, -1))[0][0] >= c:
        c = cosine_similarity(test.reshape(1, -1), compare.reshape(1, -1))[0][0]
        check = x
c
check

'-0.045824803 -0.029663097 0.019920103 -0.024606697 0.047918297 0.0122822 -0.026478902 0.0396431 0.014524999 -0.026600102'

In [124]:
querytest = """SELECT poem FROM poems WHERE vector = ?"""
dftest = pd.read_sql_query(querytest, params = [check], con=con)
dftest["poem"][0]

'Не для веселости я пью вино.\nНе для распутства пить мне суждено.\nНет, все забыть! Меня, как сам ты видишь,\nПить заставляет это лишь одно.\n'

**(1) Маленькая случайность: дорабатывала базу данных. Решила заполнить столбцы популярности нулями сразу, чтобы далее при посчёте рейтинга не страдать с Null**

In [4]:
cur.execute("""UPDATE poems SET popularity = 0""")

In [5]:
con.commit()